In [3]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.ml.feature import (
    StandardScaler, VectorAssembler, Tokenizer, StopWordsRemover,
    CountVectorizer, IDF, StringIndexer
)
from pyspark.ml.stat import Correlation
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql.functions import monotonically_increasing_id

# Start Spark session
spark = SparkSession.builder.getOrCreate()

# Load Dataset
DATASET_PATH = "/content/part-00000-tid-7038281845935297919-8101062a-5539-421c-89f8-eee1d19194bc-683-1-c000.csv"
df_clean = spark.read.option("header", True).option("inferSchema", True).csv(DATASET_PATH)
df_clean = df_clean.toDF(*[c.strip().lower().replace(" ", "_") for c in df_clean.columns])
print("\n Dataset Loaded:")
df_clean.printSchema()

# --- Feature Engineering ---

def feature_correlation(df):
    print("\n Correlation Analysis:")
    num_cols = ["year", "x_coordinate", "y_coordinate", "latitude", "longitude"]
    assembler = VectorAssembler(inputCols=num_cols, outputCol="features_corr")
    df_vec = assembler.transform(df.select(num_cols))
    corr = Correlation.corr(df_vec, "features_corr", "pearson").head()[0]
    print(corr)

def normalize_features(df):
    print("\n Standardizing features...")
    num_cols = ["year", "x_coordinate", "y_coordinate", "latitude", "longitude"]
    assembler = VectorAssembler(inputCols=num_cols, outputCol="features_unscaled")
    df = assembler.transform(df)
    scaler = StandardScaler(inputCol="features_unscaled", outputCol="features_scaled", withMean=True, withStd=True)
    model = scaler.fit(df)
    return model.transform(df).withColumn("row_id", monotonically_increasing_id())

def tokenize_and_vectorize(df):
    print("\n CountVectorizer + TF-IDF Text Vectorization...")
    df = Tokenizer(inputCol="description", outputCol="words").transform(df)
    df = StopWordsRemover(inputCol="words", outputCol="filtered_words").transform(df)
    cv = CountVectorizer(inputCol="filtered_words", outputCol="tf_features", vocabSize=1000)
    model_cv = cv.fit(df)
    df = model_cv.transform(df)
    idf = IDF(inputCol="tf_features", outputCol="tfidf_features").fit(df)
    return idf.transform(df).withColumn("row_id", monotonically_increasing_id())

def label_encode(df):
    indexer = StringIndexer(inputCol="primary_type", outputCol="label")
    return indexer.fit(df).transform(df).withColumn("row_id", monotonically_increasing_id())

# --- ML Functions ---

def train_and_evaluate(df, title="Model"):
    df = df.sample(fraction=0.1, seed=42)
    train, test = df.randomSplit([0.8, 0.2], seed=42)
    train.cache()

    rf = RandomForestClassifier(featuresCol="features", labelCol="label", seed=42)
    param_grid = (ParamGridBuilder()
                  .addGrid(rf.numTrees, [50])
                  .addGrid(rf.maxDepth, [5])
                  .build())

    evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

    cv = CrossValidator(estimator=rf,
                        estimatorParamMaps=param_grid,
                        evaluator=evaluator,
                        numFolds=2)

    print(f"Training model: {title} ... (this may still take a few minutes)")
    model = cv.fit(train)
    preds = model.transform(test)

    acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy").evaluate(preds)
    f1 = evaluator.evaluate(preds)

    print(f"\n {title}")
    print(f"    Accuracy: {acc:.4f}")
    print(f"    F1 Score: {f1:.4f}")


# --- Pipeline Execution ---

print("\n🚀 Starting Feature Analysis...")
feature_correlation(df_clean)
df_scaled = normalize_features(df_clean)
df_tfidf = tokenize_and_vectorize(df_clean)
df_labeled = label_encode(df_clean)

# --- Model A: Only Scaled Numerical Features ---
df_model_a = df_scaled.select("row_id", "features_scaled").join(df_labeled.select("row_id", "label"), on="row_id")
train_and_evaluate(df_model_a.select("label", "features_scaled").withColumnRenamed("features_scaled", "features"), "Model A: Scaled Numeric Only")

# --- Model B: Only TF-IDF Features ---
df_model_b = df_tfidf.select("row_id", "tfidf_features").join(df_labeled.select("row_id", "label"), on="row_id")
train_and_evaluate(df_model_b.select("label", "tfidf_features").withColumnRenamed("tfidf_features", "features"), "Model B: TF-IDF Text Only")

# --- Model C: Combined Features ---
df_combined = df_scaled.select("row_id", "features_scaled").join(df_tfidf.select("row_id", "tfidf_features"), on="row_id")
df_combined = df_combined.join(df_labeled.select("row_id", "label"), on="row_id")
assembler = VectorAssembler(inputCols=["features_scaled", "tfidf_features"], outputCol="features")
df_model_c = assembler.transform(df_combined)
train_and_evaluate(df_model_c.select("label", "features"), "Model C: Combined Scaled + TF-IDF")

print("\n All models trained and evaluated!")



 Dataset Loaded:
root
 |-- id: integer (nullable = true)
 |-- case_number: string (nullable = true)
 |-- date: string (nullable = true)
 |-- block: string (nullable = true)
 |-- iucr: string (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- location_description: string (nullable = true)
 |-- arrest: boolean (nullable = true)
 |-- domestic: boolean (nullable = true)
 |-- beat: integer (nullable = true)
 |-- district: integer (nullable = true)
 |-- ward: integer (nullable = true)
 |-- community_area: integer (nullable = true)
 |-- fbi_code: string (nullable = true)
 |-- x_coordinate: integer (nullable = true)
 |-- y_coordinate: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- updated_on: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- location: string (nullable = true)


🚀 Starting Feature Analysis...

 Correlation Analysis:
DenseMatrix([[ 1.     